# When Should Colombo Act? Dengue Outbreak Control Timing with a Calibrated SEIR Model

Dengue is a major public-health burden in Sri Lanka, and Colombo District reports more cases than any other district in most years. This notebook builds a seasonal SEIR (Susceptible–Exposed–Infectious–Recovered) model of dengue in Colombo, solves it with a hand-written fourth-order Runge–Kutta (RK4) method, calibrates it to weekly surveillance data with the Metropolis MCMC algorithm, and uses the fitted model to estimate how many cases a transmission-reducing intervention prevents when it starts earlier rather than later, with credible intervals from the posterior samples.

**Status:** the data, model and solver are complete; calibration, verification checks and the intervention experiments are added in later sections as the project progresses.

*Run with* **Runtime → Run all**. *Everything is downloaded from the public repository; no files or manual steps are needed.*

In [ ]:
# Setup: packages, seeds and data location.
# All of these are pre-installed on Google Colab; the install line makes local runs work too.
%pip install -q --disable-pip-version-check numpy scipy matplotlib pandas

import sys
import time
import numpy as np
import pandas as pd
import scipy
import matplotlib
import matplotlib.pyplot as plt

NOTEBOOK_START = time.perf_counter()

SEED = 20261115                # master seed: every random generator below is derived from it
rng = np.random.default_rng(SEED)

REPO_RAW = "https://raw.githubusercontent.com/IT23226814/colombo-dengue-control-timing/main"

plt.rcParams.update({"figure.figsize": (9, 4.5), "axes.grid": True, "grid.alpha": 0.3})
print(f"Python {sys.version.split()[0]} | numpy {np.__version__} | scipy {scipy.__version__} | "
      f"pandas {pd.__version__} | matplotlib {matplotlib.__version__}")
print(f"Seed: {SEED}")

## 1. Problem and question

Dengue cases in Colombo rise sharply in some years, usually after the monsoon rains. Health officials must decide **when** to start vector-control and clean-up campaigns: acting early costs resources before the outbreak is obvious, while acting late can mean thousands of extra cases.

**Question.** *If an intervention reduces dengue transmission in Colombo District by a given percentage, how many reported cases does it prevent when it starts earlier rather than later?*

The answer is given as cases prevented, with 95% credible intervals, for a range of start weeks and reduction levels, using a model calibrated to the 2026 weekly case series.

**Scope.** One district (Colombo), one season (2026, weeks 1–45), one aggregated intervention that lowers transmission by a fixed fraction from its start week. The mosquito population is not modelled explicitly in the core model.

## 2. Model, assumptions and equations

**State variables** (people): $S$ susceptible, $E$ exposed (infected, not yet infectious), $I$ infectious, $R$ recovered or immune, and $C$ the cumulative number of new symptomatic infections. Time $t$ is in **weeks**.

$$
\begin{aligned}
\frac{dS}{dt} &= -\beta(t)\,\frac{S\,I}{N}, &
\frac{dE}{dt} &= \beta(t)\,\frac{S\,I}{N} - \sigma E, &
\frac{dI}{dt} &= \sigma E - \gamma I, \\
\frac{dR}{dt} &= \gamma I, &
\frac{dC}{dt} &= \sigma E, &
N &= S + E + I + R .
\end{aligned}
$$

**Seasonal transmission with an intervention**

$$
\beta(t) = \beta_0\left[1 + a\cos\!\left(\frac{2\pi\,(t-\phi)}{52}\right)\right] u(t),
\qquad
u(t) = \begin{cases} 1 - r, & t_{\text{start}} \le t < t_{\text{end}} \\ 1, & \text{otherwise.} \end{cases}
$$

**Observation model.** New symptomatic infections in week $k$ are $C(k+1) - C(k)$; the expected number of **reported** cases is $\rho\,[C(k+1) - C(k)]$, where $\rho$ is the reporting rate.

**Basic reproduction number.** For a fully susceptible population, $R_0(t) = \beta(t)/\gamma$. If $R_0(t) < 1$ at all times, infections can only decline.

| Symbol | Meaning | Units | Value / source |
|---|---|---|---|
| $N$ | Colombo District population | people | 2,375,415 (Census of Population and Housing 2024) |
| $\beta_0$ | mean transmission rate | 1/week | estimated (MCMC) |
| $a$ | seasonal amplitude | – | estimated (MCMC), $0 \le a < 1$ |
| $\phi$ | week of peak transmission | weeks | estimated (MCMC) |
| $1/\sigma$ | intrinsic incubation period | weeks | 5.9 days = 0.843 weeks [3] |
| $1/\gamma$ | infectious (viraemic) period | weeks | 5 days = 0.714 weeks [4] |
| $\rho$ | reporting rate | – | estimated (MCMC) |
| $s_0$ | initial susceptible fraction | – | estimated (MCMC) |
| $r$, $t_{\text{start}}$ | intervention strength and start week | –, weeks | scenario inputs |

**Assumptions**
1. *Homogeneous mixing* within the district; no movement between districts.
2. *Host-only model:* transmission through *Aedes* mosquitoes, including the extrinsic incubation in the mosquito (about 8–12 days [4]), is absorbed into $\beta(t)$. This is the main structural simplification; a host–vector extension is a stretch goal.
3. *Seasonality* is a single annual cosine in $\beta$, standing in for rainfall- and temperature-driven mosquito abundance.
4. *Immunity:* only a fraction $s_0$ of the population is susceptible at the start of the year (past exposure to dengue); recovery gives immunity for the rest of the season. Serotypes are not distinguished.
5. *Reporting:* reported cases are a fixed fraction $\rho$ of symptomatic infections; reporting delays are ignored (the data already use revised weekly counts).
6. *Intervention:* any combination of measures is summarised as a fractional cut $r$ in transmission from week $t_{\text{start}}$.

**Concepts applied:** compartmental epidemic models (SEIR); numerical solution of ODEs with the classical fourth-order Runge–Kutta method and its error order; Bayesian calibration with the Metropolis MCMC algorithm; Monte Carlo propagation of parameter uncertainty into credible intervals.

## 3. Implementation

### 3.1 Data

Weekly suspected dengue cases for Colombo District, taken from the National Dengue Control Unit's *Weekly Dengue Update* reports [1], and the district population from the 2024 census [2]. How the series was extracted and checked is documented in [`data/README.md`](https://github.com/IT23226814/colombo-dengue-control-timing/blob/main/data/README.md).

*Provenance: data-loading code written with an AI coding agent.*

In [ ]:
cases = pd.read_csv(f"{REPO_RAW}/data/colombo_weekly_cases.csv", parse_dates=["week_start", "week_end"])
population = pd.read_csv(f"{REPO_RAW}/data/colombo_population.csv")
N_COLOMBO = int(population.loc[0, "population"])          # people

obs = cases[cases.year == 2026].sort_values("week").reset_index(drop=True)
WEEKS = len(obs)                                           # number of observed weeks
y_obs = obs["cases"].to_numpy()                            # reported cases per week

# Sanity checks on the input data
assert N_COLOMBO == 2_375_415
assert obs["week"].tolist() == list(range(1, WEEKS + 1)), "2026 weeks must be consecutive from week 1"
assert (y_obs >= 0).all() and not obs["cases"].isna().any()

print(f"Population N = {N_COLOMBO:,}")
print(f"2026 data: weeks 1–{WEEKS} ({obs.week_start.iloc[0].date()} to {obs.week_end.iloc[-1].date()}), "
      f"{y_obs.sum():,} reported cases, peak {y_obs.max():,} in week {obs.week[y_obs.argmax()]}")

fig, ax = plt.subplots()
for year, colour in [(2025, "0.55"), (2026, "C3")]:
    d = cases[cases.year == year].set_index("week")["cases"].reindex(range(1, 53))  # gaps stay empty
    ax.plot(d.index, d.values, "o-", ms=3, color=colour, label=str(year))
ax.set(xlabel="Week of year", ylabel="Reported cases per week", title="Weekly dengue cases, Colombo District", xlim=(0, 53))
ax.legend(frameon=False)
plt.show()

### 3.2 SEIR model and RK4 solver

The model is written from scratch: the right-hand side `rhs`, a classical RK4 step `rk4_step` (four stages, local error $O(h^5)$, global error $O(h^4)$), and `simulate`, which advances the state with a fixed step of $h = 1/7$ week (one day). The cumulative compartment $C$ gives exact weekly incidence as a difference of two RK4 states. Every parameter may be a NumPy array, so many parameter sets (e.g. posterior samples) are simulated in one call.

The same code is kept in [`src/seir.py`](https://github.com/IT23226814/colombo-dengue-control-timing/blob/main/src/seir.py), where an automated test suite also runs against it; a test checks that this cell and the file stay identical.

*Provenance: written with an AI coding agent; see the project's AI-use record.*

In [ ]:
"""Seasonal SEIR model of dengue transmission, solved with a hand-written RK4.

Units used throughout
---------------------
time            weeks (t = 0 is the start of the first simulated week)
rates           1/week (beta0, sigma, gamma)
compartments    people (S, E, I, R and N)
C               cumulative new symptomatic infections (people), dC/dt = sigma*E

Model
-----
    dS/dt = -beta(t) S I / N
    dE/dt =  beta(t) S I / N - sigma E
    dI/dt =  sigma E - gamma I
    dR/dt =  gamma I
    dC/dt =  sigma E                      (counts every E -> I transition)

    beta(t) = beta0 * [1 + amp * cos(2 pi (t - phase) / period)] * u(t)
    u(t)    = 1 - reduction  while start <= t < end  (intervention), else 1

Weekly new infections in week k (from t = k to k + 1) are C(k+1) - C(k), and
the expected *reported* cases are rho times that.

This is a host-only model: the mosquito stage is not modelled explicitly, so
transmission through mosquitoes is absorbed into beta(t).

Every parameter may be a float or a NumPy array. Arrays are broadcast against
each other, so one call can run many parameter sets (e.g. MCMC samples) at once.
"""

from __future__ import annotations

from dataclasses import dataclass, fields, replace

import numpy as np

DAYS_PER_WEEK = 7.0

# Literature defaults for the natural history (converted to 1/week):
#   mean intrinsic incubation period 5.9 days  [Chan & Johansson 2012, PLoS ONE 7(11):e50972,
#                                               doi:10.1371/journal.pone.0050972]
#   human viraemic (infectious) period ~5 days [WHO dengue fact sheet: "about 4-5 days"]
SIGMA_DEFAULT = DAYS_PER_WEEK / 5.9
GAMMA_DEFAULT = DAYS_PER_WEEK / 5.0


@dataclass(frozen=True)
class Params:
    """Model parameters. Floats or broadcastable NumPy arrays."""

    N: float                      # population (people)
    beta0: float                  # mean transmission rate (1/week)
    amp: float = 0.0              # seasonal amplitude, 0 <= amp < 1 (dimensionless)
    phase: float = 0.0            # week of peak transmission (weeks)
    period: float = 52.0          # length of the seasonal cycle (weeks)
    sigma: float = SIGMA_DEFAULT  # 1 / incubation period (1/week)
    gamma: float = GAMMA_DEFAULT  # 1 / infectious period (1/week)
    rho: float = 1.0              # reporting rate: reported / symptomatic infections
    s0: float = 1.0               # initial susceptible fraction of N
    e0: float = 0.0               # initial exposed (people)
    i0: float = 1.0               # initial infectious (people)
    start: float = np.inf         # intervention start (weeks)
    end: float = np.inf           # intervention end (weeks)
    reduction: float = 0.0        # fractional cut in beta during the intervention

    def with_(self, **changes) -> "Params":
        """Return a copy with some parameters changed."""
        return replace(self, **changes)

    def arrays(self) -> dict[str, np.ndarray]:
        """All parameters as NumPy arrays broadcast to one common shape."""
        names = [f.name for f in fields(self)]
        vals = np.broadcast_arrays(*[np.asarray(getattr(self, n), dtype=float) for n in names])
        return dict(zip(names, vals))


def control(t: float, p: dict[str, np.ndarray]) -> np.ndarray:
    """Intervention multiplier u(t): 1 - reduction while start <= t < end, else 1."""
    active = (t >= p["start"]) & (t < p["end"])
    return np.where(active, 1.0 - p["reduction"], 1.0)


def beta(t: float, p: dict[str, np.ndarray], u: np.ndarray | None = None) -> np.ndarray:
    """Transmission rate at time t (1/week). u is the intervention multiplier;
    if None it is evaluated at t."""
    seasonal = p["beta0"] * (1.0 + p["amp"] * np.cos(2.0 * np.pi * (t - p["phase"]) / p["period"]))
    return seasonal * (control(t, p) if u is None else u)


def basic_reproduction_number(t: float, params: Params) -> np.ndarray:
    """Instantaneous R0(t) = beta(t) / gamma for a fully susceptible population."""
    p = params.arrays()
    return beta(t, p) / p["gamma"]


def rhs(t: float, y: np.ndarray, p: dict[str, np.ndarray], u: np.ndarray | None = None) -> np.ndarray:
    """Right-hand side f(t, y); y has shape (5, ...) = (S, E, I, R, C)."""
    S, E, I, R, C = y
    infection = beta(t, p, u) * S * I / p["N"]
    incidence = p["sigma"] * E
    recovery = p["gamma"] * I
    return np.stack([-infection, infection - incidence, incidence - recovery, recovery, incidence])


def rk4_step(f, t: float, y: np.ndarray, h: float, *args) -> np.ndarray:
    """One classical 4th-order Runge-Kutta step of size h for y' = f(t, y, *args)."""
    k1 = f(t, y, *args)
    k2 = f(t + h / 2, y + h / 2 * k1, *args)
    k3 = f(t + h / 2, y + h / 2 * k2, *args)
    k4 = f(t + h, y + h * k3, *args)
    return y + h / 6 * (k1 + 2 * k2 + 2 * k3 + k4)


def initial_state(p: dict[str, np.ndarray]) -> np.ndarray:
    """(S, E, I, R, C) at t = 0. People not susceptible, exposed or infectious start in R."""
    S = p["s0"] * p["N"] - p["e0"] - p["i0"]
    R = p["N"] - S - p["e0"] - p["i0"]
    if np.any(S < 0):
        raise ValueError("s0 * N must be at least e0 + i0")
    return np.stack([S, p["e0"], p["i0"], R, np.zeros_like(S)])


def simulate(params: Params, weeks: int, steps_per_week: int = 7) -> dict[str, np.ndarray]:
    """Integrate the model for `weeks` weeks with fixed RK4 steps of 1/steps_per_week weeks.

    Returns a dict with
      t                 (weeks*steps_per_week + 1,) time grid in weeks
      y                 (len(t), 5, ...) states S, E, I, R, C on the grid
      weekly_infections (weeks, ...) new symptomatic infections in each week
      weekly_reported   (weeks, ...) expected reported cases = rho * weekly_infections

    The intervention multiplier u is held constant over each step, at its value
    at the step midpoint, so the right-hand side is smooth within every step.
    Start/end times on the step grid (e.g. whole weeks) are therefore handled
    exactly and keep full 4th-order accuracy.
    """
    p = params.arrays()
    h = 1.0 / steps_per_week
    n = weeks * steps_per_week
    y = initial_state(p)
    out = np.empty((n + 1,) + y.shape)
    out[0] = y
    for k in range(n):
        t = k * h
        y = rk4_step(rhs, t, y, h, p, control(t + h / 2, p))
        out[k + 1] = y
    C_weekly = out[::steps_per_week, 4]
    infections = np.diff(C_weekly, axis=0)
    return {
        "t": np.arange(n + 1) * h,
        "y": out,
        "weekly_infections": infections,
        "weekly_reported": p["rho"] * infections,
    }

**Example run (illustrative parameters, not fitted).** The values below only show that the model produces a seasonal outbreak and that an intervention lowers it; the calibrated values come from Section 3.3.

In [ ]:
example = Params(N=N_COLOMBO, beta0=2.8, amp=0.5, phase=22.0, rho=0.1, s0=0.5, e0=1500.0, i0=1500.0)
weeks = np.arange(1, WEEKS + 1)

baseline = simulate(example, WEEKS)
intervention = simulate(example.with_(start=18, reduction=0.25), WEEKS)

fig, ax = plt.subplots()
ax.plot(weeks, baseline["weekly_reported"], label="no intervention")
ax.plot(weeks, intervention["weekly_reported"], label="25% cut in transmission from week 18")
ax.axvline(18, color="0.5", ls=":")
ax.set(xlabel="Week of 2026", ylabel="Expected reported cases per week", title="Illustrative model run (not fitted)")
ax.legend(frameon=False)
plt.show()

prevented = baseline["weekly_reported"].sum() - intervention["weekly_reported"].sum()
print(f"Illustrative only: {prevented:,.0f} reported cases prevented over weeks 1–{WEEKS}")

### 3.3 Calibration with Metropolis MCMC

*To be added: likelihood for weekly counts, priors, a random-walk Metropolis sampler with several chains, convergence diagnostics and a posterior predictive check against the 2026 data.*

## 4. Verification

*To be added: assert-based checks that a reader can see pass — conservation of $S+E+I+R$, the $R_0<1$ limiting case, the fourth-order convergence of RK4 on a log–log plot, and recovery of known parameters from synthetic data by the MCMC sampler.*

## 5. Experiments and results

*To be added: cases prevented for a grid of intervention start weeks and strengths, computed over posterior samples and reported as medians with 95% credible intervals, followed by a sensitivity analysis of the key inputs.*

## 6. Discussion and limitations

*To be completed with the results.* The main limitations already known are the host-only structure (no explicit mosquito population), a single sinusoidal seasonal pattern, homogeneous mixing across the whole district, and suspected (not laboratory-confirmed) case counts with a constant reporting rate.

**Web demo reference value:** *to be stated once the calibrated results are available.*

## 7. References

1. National Dengue Control Unit, Ministry of Health, Sri Lanka, "Weekly Dengue Update," 2026, weeks 1–45. [Online]. Available: https://www.dengue.health.gov.lk/weekly-report/
2. Department of Census and Statistics, Sri Lanka, "Census of Population and Housing 2024," final report, 2026. [Online]. Available: https://www.statistics.gov.lk/Blog/Blog_details
3. M. Chan and M. A. Johansson, "The incubation periods of dengue viruses," *PLoS ONE*, vol. 7, no. 11, p. e50972, 2012, doi: 10.1371/journal.pone.0050972.
4. World Health Organization, "Dengue and severe dengue," fact sheet. [Online]. Available: https://www.who.int/news-room/fact-sheets/detail/dengue-and-severe-dengue

In [ ]:
elapsed = time.perf_counter() - NOTEBOOK_START
print(f"Notebook ran in {elapsed:.1f} s")
assert elapsed < 600, "Run all should finish within 10 minutes"